<a href="https://colab.research.google.com/github/sana200420/naari-ai/blob/sana%2Ftest-protection/retrieval/scripts/embed_english_kb.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Embed the English KB (Lever 4)

Adds the English side of the cross-lingual dual index: 1,999 English rows
(Sabiha/Sana/Mahnoor/Tooba's translations, only `id=2000` still untranslated)
into the **same** `naari_ai_kb` Qdrant collection already holding the 2,000
Sindhi points.

**Important:** English points reuse the same `answer_id` values as their
Sindhi twins (that's the whole point of Lever 4 -- `answer_id` is the join key
across languages). But Qdrant point IDs must be unique *within* a collection,
and the Sindhi points already occupy point IDs 1-2000. So English points are
stored at point ID `10000 + answer_id`, with the real `answer_id` kept in the
payload as the actual join key, and `lang: "en"` to tell them apart. Nothing
about the Sindhi points changes.

Same workflow as `embed_and_index.ipynb`: mount Drive for checkpointing,
**Runtime -> Change runtime type -> T4 GPU**, run all, paste Qdrant
credentials when prompted.

In [11]:
!pip install -q qdrant-client FlagEmbedding

In [12]:
from google.colab import drive
drive.mount("/content/drive")

import os
CHECKPOINT_DIR = "/content/drive/MyDrive/naari_ai_checkpoints"
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
EMBED_CHECKPOINT = f"{CHECKPOINT_DIR}/embed_english_checkpoint.pkl"
UPSERT_CHECKPOINT = f"{CHECKPOINT_DIR}/upsert_english_progress.json"
print("checkpoint dir:", CHECKPOINT_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
checkpoint dir: /content/drive/MyDrive/naari_ai_checkpoints


In [13]:
!rm -rf naari-ai
!git clone --branch sana/test-protection --depth 1 https://github.com/sana200420/naari-ai.git
%cd naari-ai

import sys
sys.path.insert(0, ".")

from retrieval.normalize import normalize_sd

print("cloned + imported OK")

Cloning into 'naari-ai'...
remote: Enumerating objects: 80, done.
remote: Counting objects: 100% (80/80), done.
remote: Compressing objects: 100% (73/73), done.
remote: Total 80 (delta 4), reused 62 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (80/80), 698.70 KiB | 6.18 MiB/s, done.
Resolving deltas: 100% (4/4), done.
/content/naari-ai/naari-ai
cloned + imported OK


In [14]:
import csv

EN_KB_PATH = "knowledge_base/Womens_Health_KB_English - 2000_final.csv"
with open(EN_KB_PATH, encoding="utf-8", newline="") as f:
    rows = list(csv.DictReader(f))

print(f"{len(rows)} English rows loaded (expect 1999 -- id=2000 has no translation yet)")
print(rows[0])

assert len(set(r["id"] for r in rows)) == len(rows), "duplicate ids found"
assert all(r["question"].strip() and r["answer"].strip() for r in rows), "blank field found"

1999 English rows loaded (expect 1999 -- id=2000 has no translation yet)
{'id': '1', 'category': 'Menstrual Health & Periods', 'sub_category': "Cycle Basics & What's Normal", 'question': 'What is a menstrual cycle?', 'answer': "A menstrual cycle is the monthly hormonal process in which the uterine lining builds up and then sheds as a period if pregnancy doesn't occur. It is counted from the first day of one period to the first day of the next.", 'source': 'WHO - https://www.who.int/news-room/fact-sheets/detail/menstrual-health-and-hygiene', 'review_tier': 'B'}


## Load bge-m3 and embed all English questions (dense + sparse)

In [15]:
from FlagEmbedding import BGEM3FlagModel

model = BGEM3FlagModel("BAAI/bge-m3", use_fp16=True)
print("model loaded")

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 30 files:   0%|          | 0/30 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

model loaded


In [16]:
# Every embed goes through normalize_sd() -- it's a no-op on most English text
# (no Arabic-script mapping applies) except whitespace collapse and case fold,
# but it's still the one gate everything embeds through, per retrieval/embed.py's
# design: nothing bypasses normalisation, regardless of language.
normalised_questions = [normalize_sd(row["question"]) for row in rows]
assert len(normalised_questions) == len(rows)
print(normalised_questions[0])

what is a menstrual cycle?


In [17]:
import pickle

BATCH = 64

if os.path.exists(EMBED_CHECKPOINT):
    with open(EMBED_CHECKPOINT, "rb") as f:
        checkpoint = pickle.load(f)
    dense_vecs = checkpoint["dense_vecs"]
    sparse_vecs = checkpoint["sparse_vecs"]
    start = len(dense_vecs)
    print(f"resuming from checkpoint: {start}/{len(normalised_questions)} already embedded")
else:
    dense_vecs = []
    sparse_vecs = []
    start = 0
    print("no checkpoint found, starting fresh")

for i in range(start, len(normalised_questions), BATCH):
    batch = normalised_questions[i:i + BATCH]
    out = model.encode(batch, return_dense=True, return_sparse=True, return_colbert_vecs=False)
    dense_vecs.extend(out["dense_vecs"])
    sparse_vecs.extend(out["lexical_weights"])

    with open(EMBED_CHECKPOINT, "wb") as f:
        pickle.dump({"dense_vecs": dense_vecs, "sparse_vecs": sparse_vecs}, f)

    print(f"embedded {min(i + BATCH, len(normalised_questions))}/{len(normalised_questions)} (checkpoint saved)", end="\r")

print(f"\ndone. dense dim: {len(dense_vecs[0])}")
assert len(dense_vecs) == len(sparse_vecs) == len(normalised_questions)

resuming from checkpoint: 1999/1999 already embedded

done. dense dim: 1024


## Connect to Qdrant and upsert into the existing collection

In [18]:
from getpass import getpass
from qdrant_client import QdrantClient, models

QDRANT_URL = getpass("Qdrant cluster URL: ")
QDRANT_API_KEY = getpass("Qdrant API key: ")

client = QdrantClient(url=QDRANT_URL, api_key=QDRANT_API_KEY)

COLLECTION = "naari_ai_kb"
assert client.collection_exists(COLLECTION), "naari_ai_kb doesn't exist yet -- run embed_and_index.ipynb first"

before_count = client.get_collection(COLLECTION).points_count
print(f"collection exists, currently holds {before_count} points (expect 2000, the Sindhi side)")

Qdrant cluster URL: ··········
Qdrant API key: ··········
collection exists, currently holds 3999 points (expect 2000, the Sindhi side)


In [19]:
import json

ENGLISH_POINT_ID_OFFSET = 10000  # avoids colliding with Sindhi points at id 1-2000

def to_sparse_vector(lexical_weights: dict) -> models.SparseVector:
    indices = [int(k) for k in lexical_weights.keys()]
    values = [float(v) for v in lexical_weights.values()]
    return models.SparseVector(indices=indices, values=values)

points = []
for row, dense, sparse in zip(rows, dense_vecs, sparse_vecs):
    answer_id = int(row["id"])
    points.append(
        models.PointStruct(
            id=ENGLISH_POINT_ID_OFFSET + answer_id,
            vector={
                "dense": dense.tolist(),
                "sparse": to_sparse_vector(sparse),
            },
            payload={
                "answer_id": answer_id,
                "category": row["category"],
                "sub_category": row["sub_category"],
                "question": row["question"],
                "answer": row["answer"],
                "source": row["source"],
                "review_tier": row["review_tier"],
                "lang": "en",
            },
        )
    )

if os.path.exists(UPSERT_CHECKPOINT):
    with open(UPSERT_CHECKPOINT) as f:
        upserted_count = json.load(f)["upserted"]
    print(f"resuming upload from checkpoint: {upserted_count}/{len(points)} already upserted")
else:
    upserted_count = 0
    print("no upload checkpoint found, starting fresh")

UPSERT_BATCH = 128
for i in range(upserted_count, len(points), UPSERT_BATCH):
    batch = points[i:i + UPSERT_BATCH]
    client.upsert(collection_name=COLLECTION, points=batch)
    upserted_count = min(i + UPSERT_BATCH, len(points))

    with open(UPSERT_CHECKPOINT, "w") as f:
        json.dump({"upserted": upserted_count}, f)

    print(f"upserted {upserted_count}/{len(points)} (checkpoint saved)", end="\r")

print()
after_count = client.get_collection(COLLECTION).points_count
print(f"collection now holds {after_count} points (expect {before_count} + {len(points)} = {before_count + len(points)})")
assert after_count == before_count + len(points)

resuming upload from checkpoint: 1999/1999 already upserted

collection now holds 3999 points (expect 3999 + 1999 = 5998)


AssertionError: 

## Sanity check -- cross-lingual query

In [20]:
# Qdrant needs an explicit index before you can filter by a payload field --
client.create_payload_index(
    collection_name=COLLECTION,
    field_name="lang",
    field_schema=models.PayloadSchemaType.KEYWORD,
)

test_query = normalize_sd("What is a normal menstrual cycle?")
q_out = model.encode([test_query], return_dense=True, return_sparse=False, return_colbert_vecs=False)
q_vec = q_out["dense_vecs"][0].tolist()

hits = client.query_points(
    collection_name=COLLECTION,
    query=q_vec,
    using="dense",
    limit=5,
    with_payload=True,
    query_filter=models.Filter(must=[models.FieldCondition(key="lang", match=models.MatchValue(value="en"))]),
).points

for h in hits:
    print(f"{h.score:.3f}  answer_id={h.payload['answer_id']}  {h.payload['question']}")

0.888  answer_id=2  How long does a normal menstrual cycle last?
0.876  answer_id=1  What is a menstrual cycle?
0.850  answer_id=33  What counts as 'normal' menstrual flow overall?
0.780  answer_id=6  What are the main phases of the menstrual cycle?
0.758  answer_id=1677  How many phases does the menstrual cycle have?


## Next steps

- Collection should now hold 3999 points (2000 Sindhi + 1999 English) -- not
  the full 4000 target yet, since `id=2000` (the row added during the KB
  cleanup) has no English translation.
- Lever 4's actual cross-lingual cascade (translate Sindhi query -> English
  via local NLLB, only when the Sindhi leg is uncertain) is still separate
  work -- this notebook only gets the English data *into* the index, it
  doesn't wire up the query-time translation path.
- Once done, delete `embed_english_checkpoint.pkl` and
  `upsert_english_progress.json` from Drive so a future rerun doesn't
  resume from stale data.